In [3]:
import numpy as np
import matplotlib.pyplot as plt
import math
import string
import scipy.stats as stats
from matplotlib.ticker import ScalarFormatter

In [2]:
### FUNCTIONS FOR FREQUENCY CALCULATION ###

#calculate the amount of distinguishable permutations:
def distinguishable_permutations(array):
    uniques = (np.unique(array))
    total = math.factorial(len(array))
    for u in uniques:
        occ = (list(array).count(u))
        if total//math.factorial(int(occ)) > sys.float_info.max:
            total = total//math.factorial(int(occ))
        else:
            total = total/math.factorial(int(occ))
    return(total)

#calculate the amount of indistinguishable permutations:
def indistinguishable_permutations(array):
    total = math.factorial(int(np.sum(array)))
    for a in array:
        if total//math.factorial(int(a)) > sys.float_info.max:
            total = total//math.factorial(int(a))
        else:
            total = total/math.factorial(int(a))
    return(total)

def frequency(array):
    return (int(indistinguishable_permutations(array))*int(distinguishable_permutations(array)))/(len(array)**int(np.sum(array)))

In [ ]:
### SHANNON INDEX AND COSINE SIMILARITY ###
def shannon(data):
    zeros = np.prod(np.shape(data)) - np.count_nonzero(np.array(data, dtype = np.uint8))
    N = np.sum(data)+zeros
    P = data/N
    H = 0
    S = 0
    
    for value in np.ravel(P):
        if value!= 0:
            H -= value*np.log(value)
            S += 1
    S += 1
    return H, S

def cosine_similarity(A,B):
    return np.sum(A*B)/(np.sqrt(np.sum(A**2))*np.sqrt(np.sum(B**2)))

def similarity_coef(data):
    """ axis0 = materials, 
        axis1 = phenomena"""
    sumSubjects = np.sum(data, axis = 0)
    sumMaterials = np.sum(data, axis = 1)
    
    simMaterials = cosine_similarity(sumMaterials, np.ones(np.shape(sumMaterials)))
    simSubjects = cosine_similarity(sumSubjects, np.ones(np.shape(sumSubjects)))
    
    simAll = cosine_similarity(data, np.ones(np.shape(data)))
    
    return simMaterials, simSubjects, simAll

In [ ]:
### PLOTTING FUNCTIONS ###
def plot_heatmap(ax, data, materials, subjects, maxRange):
    norm = plt.Normalize(np.log(1), np.log(maxRange + 1))
    ax.set_xticks(np.arange(len(subjects)))
    ax.set_xticklabels(subjects, minor=False, rotation=90)
    #ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)
    ax.set_yticks(np.arange(len(materials)))
    ax.set_yticklabels(materials, minor=False, rotation=0)
    mappable = ax.imshow(np.log(data+1), cmap = "plasma", norm = norm)
    print(data)
    print(np.sum(data))
    #print(mappable)
    return mappable

def sort_heatmap(data):
    sorted_labels0 = np.argsort(np.sum(data, axis = 0))
    sorted_labels0_2d = np.tile(sorted_labels0, np.shape(data)[0])
    sorted_labels0_2d = np.reshape(sorted_labels0_2d, np.shape(data))
    
    sorted_labels1 = np.argsort(np.sum(np.take_along_axis(data, sorted_labels0_2d, axis = 1), axis = 1))
    sorted_labels1_2d = np.tile(sorted_labels1, np.shape(data)[1])
    sorted_labels1_2d = np.transpose(np.reshape(sorted_labels1_2d, np.shape(data)[::-1]))
    return sorted_labels0, sorted_labels0_2d, sorted_labels1, sorted_labels1_2d

def plot_sorted_heatmap(ax, data, materials,subjects, maxRange = 46):
        """plot a heatmap of all material and phenonomena combinations, with the materials (vertically) and phenomena (horizontally) sorted from least to most.
           ax = [matplotlib.figure] the figure to which the heatmap is added
           data = [2D numpy array] array of with the occurences of each material and phenomenon combination
           materials = [1D numpy array] an array of strings with the names of all materials
           subjects = [1D numpy array] an array of strings with the names of all phenomena
           maxRange = [int] maximum value in data"""
    sorted_labels0, sorted_labels0_2d, sorted_labels1, sorted_labels1_2d = sort_heatmap(data)
    mappable = plot_heatmap(ax, np.take_along_axis(np.take_along_axis(data, sorted_labels0_2d, axis = 1), sorted_labels1_2d, axis = 0), np.take_along_axis(materials, sorted_labels1, axis = 0), np.take_along_axis(subjects, sorted_labels0, axis = 0), maxRange)
    return mappable

In [ ]:
### FOR QUESTIONS RECARDING THE CODE CONTACT ANB@CSWALCHEREN.NL ###